In [34]:
import importlib
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from unittest.mock import patch

import pandas as pd
from IPython.display import display

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'src').is_dir():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate project root containing src/')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import emoji_norm, formatters, noise_cleaner, pipeline, quality_filter, split_dataset, unicode_norm, vocab_norm

emoji_norm = importlib.reload(emoji_norm)
formatters = importlib.reload(formatters)
noise_cleaner = importlib.reload(noise_cleaner)
quality_filter = importlib.reload(quality_filter)
unicode_norm = importlib.reload(unicode_norm)
vocab_norm = importlib.reload(vocab_norm)
pipeline = importlib.reload(pipeline)
split_dataset = importlib.reload(split_dataset)


In [35]:
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)


In [36]:
def lowercase_series(series: pd.Series) -> pd.Series:
    return series.map(lambda value: value.lower() if isinstance(value, str) else value)


def stepwise_preview(series: pd.Series, min_chars: int = 5) -> pd.DataFrame:
    preview = pd.DataFrame({'raw': series})
    preview['unicode'] = unicode_norm.normalize_series(preview['raw'])
    preview['noise'] = noise_cleaner.normalize_series(preview['unicode'])
    preview['emoji'] = emoji_norm.normalize_series(preview['noise'])
    preview['vocab'] = vocab_norm.normalize_series(preview['emoji'])
    preview['format'] = formatters.normalize_series(preview['vocab'])
    preview['clean'] = lowercase_series(preview['format'])
    preview['meaningful'] = preview['clean'].map(lambda value: quality_filter.is_meaningful_text(value, min_chars=min_chars))
    preview['group_key'] = preview['clean'].map(quality_filter.normalize_for_duplicate)
    return preview


def summarize_frame(df: pd.DataFrame, text_column: str = 'content') -> pd.DataFrame:
    summary = {'rows': len(df), 'columns': len(df.columns)}
    if text_column in df.columns:
        summary['missing_text'] = int(df[text_column].isna().sum())
        summary['duplicate_normalized_texts'] = int(df[text_column].map(quality_filter.normalize_for_duplicate).duplicated().sum())
    return pd.DataFrame([summary])


## 1. Pipeline call order

This reproduces `test_clean_text_series_applies_steps_in_order`.


In [37]:
calls: list[str] = []


def make_step(name: str):
    def step(series: pd.Series) -> pd.Series:
        calls.append(name)
        return pd.Series([f'{series.iloc[0]}->{name}'], index=series.index)

    return step


with patch.object(pipeline.unicode_norm, 'normalize_series', make_step('unicode')):
    with patch.object(pipeline.noise_cleaner, 'normalize_series', make_step('noise')):
        with patch.object(pipeline.emoji_norm, 'normalize_series', make_step('emoji')):
            with patch.object(pipeline.vocab_norm, 'normalize_series', make_step('vocab')):
                with patch.object(pipeline.formatters, 'normalize_series', make_step('format')):
                    ordered_result = pipeline.clean_text_series(pd.Series(['start']))

display(pd.DataFrame({'result': ordered_result}))
print(calls)

assert ordered_result.tolist() == ['start->unicode->noise->emoji->vocab->format']
assert calls == ['unicode', 'noise', 'emoji', 'vocab', 'format']


,result
0,start->unicode->noise->emoji->vocab->format


['unicode', 'noise', 'emoji', 'vocab', 'format']


## 2. Module smoke tests

These cells mirror the lower-level unit tests so you can see the exact failure point.


In [38]:
# Unicode
with patch.object(unicode_norm, 'fix_text', lambda text: text):
    raw = 'Cafe\u0301\x00\tA\u200bB\nC\u200dD'
    assert unicode_norm.normalize_unicode(raw) == 'Caf\u00e9\tAB\nC\u200dD'

assert unicode_norm.normalize_unicode(None) is None
assert unicode_norm.normalize_unicode(float('nan')) is None

unicode_frame = pd.DataFrame({'content': ['A' + chr(0) + 'B', None], 'other': [1, 2]})
unicode_normalized = unicode_norm.normalize_dataframe(unicode_frame, 'content', output_column='clean')
display(unicode_normalized)

assert unicode_normalized['clean'].iloc[0] == 'AB'
assert pd.isna(unicode_normalized['clean'].iloc[1])
assert unicode_frame['content'].iloc[0] == 'A' + chr(0) + 'B'
assert pd.isna(unicode_frame['content'].iloc[1])
assert 'clean' not in unicode_frame.columns

# Noise
assert noise_cleaner.strip_html('<div>Hello <b>world</b></div>') == 'Hello world'
noise_text = '<p>Visit https://example.com or email me@example.com or call +84 912 345 678.</p>'
assert noise_cleaner.normalize_noise(noise_text) == 'Visit __url__ or email __email__ or call +__phone__.'

noise_series = pd.Series(['Hello <b>world</b>', None])
noise_normalized = noise_cleaner.normalize_series(noise_series)
assert noise_normalized.iloc[0] == 'Hello world'
assert pd.isna(noise_normalized.iloc[1])


,content,other,clean
0,A B,1,AB
1,NaN,2,NaN


In [39]:
# Emoji, vocab and formatters
assert emoji_norm.demojize_text('Hello :V. world :(((') == 'Hello :V. world :((('

with patch.object(emoji_norm, 'EMOJI_MAP', {}):
    assert emoji_norm.demojize_text('Nice \U0001F600') == 'Nice emoji_grinning_face'

with patch.object(emoji_norm, 'EMOJI_MAP', {'grinning_face': 'mat_cuoi'}):
    assert emoji_norm.demojize_text('Nice \U0001F600') == 'Nice mat_cuoi'

with patch.object(emoji_norm, 'EMOJI_MAP', {}):
    emoji_series = pd.Series(['A \U0001F600', None])
    emoji_normalized = emoji_norm.normalize_series(emoji_series)
    assert emoji_normalized.iloc[0] == 'A emoji_grinning_face'
    assert pd.isna(emoji_normalized.iloc[1])

with patch.object(vocab_norm, 'VOCAB_MAP', {'ko': 'khong'}):
    assert vocab_norm.normalize_vocab('ko') == 'khong'

assert vocab_norm.normalize_vocab('ngonnnn qua hayyyy') == 'ngon qua hayy'

with patch.object(vocab_norm, 'VOCAB_MAP', {}):
    vocab_series = pd.Series(['ngonnnn', None])
    vocab_normalized = vocab_norm.normalize_series(vocab_series)
    assert vocab_normalized.iloc[0] == 'ngon'
    assert pd.isna(vocab_normalized.iloc[1])

assert formatters.normalize_format('Xin\u200b chao!!!   ban???\n') == 'Xin chao!! ban??'
assert formatters.normalize_format(None) is None

format_series = pd.Series(['A\u200bB', None])
format_normalized = formatters.normalize_series(format_series)
assert format_normalized.iloc[0] == 'AB'
assert pd.isna(format_normalized.iloc[1])


## 3. Stepwise preview

Use this to inspect how one text changes at each cleaning stage.


In [40]:
sample_texts = pd.Series(
    [
        'Cafe\u0301\x00\tA\u200bB\nC\u200dD',
        '<p>Visit https://example.com or email me@example.com or call +84 912 345 678.</p>',
        'Hello :V. world :(((',
        'Nice \U0001F600',
        'ngonnnn qua hayyyy',
        'Xin\u200b chao!!!   ban???\n',
        'null',
        None,
    ],
    name='raw',
)

preview = stepwise_preview(sample_texts, min_chars=5)
display(preview)

assert preview.loc[2, 'clean'] == 'hello :v. world :((('
assert preview.loc[4, 'clean'] == 'ngon qua hayy'
assert preview.loc[5, 'clean'] == 'xin chao!! ban??'
assert not preview.loc[6, 'meaningful']


,raw,unicode,noise,emoji,vocab,format,clean,meaningful,group_key
0,Café \tA​B\nC‍D,Café\tAB\nC‍D,Café\tAB\nC‍D,Café AB C‍D,Café AB c‍d,Café AB cd,café ab cd,True,café ab cd
1,<p>Visit https://example.com or email me@example.com or call +84 912 345 678.</p>,<p>Visit https://example.com or email me@example.com or call +84 912 345 678.</p>,Visit __url__ or email __email__ or call +__phone__.,Visit __url__ or email __email__ or call +__phone__.,Visit __url__ or email __email__ or call +__phone__.,Visit __url__ or email __email__ or call +__phone__.,visit __url__ or email __email__ or call +__phone__.,True,visit __url__ or email __email__ or call +__phone__.
2,Hello :V. world :(((,Hello :V. world :(((,Hello :V. world :(((,Hello :V. world :(((,Hello :v. world :(((,Hello :v. world :(((,hello :v. world :(((,True,hello :v. world :(((
3,Nice 😀,Nice 😀,Nice 😀,Nice vui_vẻ,Nice vui_vẻ,Nice vui_vẻ,nice vui_vẻ,True,nice vui_vẻ
4,ngonnnn qua hayyyy,ngonnnn qua hayyyy,ngonnnn qua hayyyy,ngonnnn qua hayyyy,ngon qua hayy,ngon qua hayy,ngon qua hayy,True,ngon qua hayy
5,Xin​ chao!!! ban???\n,Xin chao!!! ban???\n,Xin chao!!! ban???\n,Xin chao!!! ban???,Xin chao!!! ban???,Xin chao!! ban??,xin chao!! ban??,True,xin chao!! ban??
6,null,null,null,null,null,null,null,False,null
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,


## 4. Quality filter

This reproduces `test_quality_filter.py` and shows the effect of `drop_noise_rows`.


In [41]:
assert quality_filter.normalize_for_duplicate('  Xin   Chao\n') == 'xin chao'
assert not quality_filter.is_meaningful_text('null', min_chars=5)
assert not quality_filter.is_meaningful_text('123456', min_chars=5)
assert not quality_filter.is_meaningful_text('!!!', min_chars=5)
assert quality_filter.is_meaningful_text('meaningful text', min_chars=5)
assert quality_filter.is_digit_only('123456')
assert quality_filter.is_symbol_only('!!!')

noise_frame = pd.DataFrame(
    {
        'content': [
            'Hello world',
            'hello   world',
            '1234567890',
            '!!!???',
            'meaningful text',
        ],
        'label': [1, 2, 3, 4, 5],
    }
)

filtered = quality_filter.drop_noise_rows(
    noise_frame,
    text_column='content',
    min_chars=5,
    drop_duplicates=True,
)

display(noise_frame)
display(filtered)

assert filtered['content'].tolist() == ['Hello world', 'meaningful text']
assert filtered['label'].tolist() == [1, 5]
assert filtered.index.tolist() == [0, 1]


,content,label
0,Hello world,1
1,hello world,2
2,1234567890,3
3,!!!???,4
4,meaningful text,5


,content,label
0,Hello world,1
1,meaningful text,5


## 5. DataFrame and file round-trip

This reproduces `test_preprocess_dataframe_keeps_raw_and_respects_keep_columns` and `test_preprocess_file_round_trips_csv`.


In [42]:
frame = pd.DataFrame(
    {
        'review_id': [1, 2],
        'content': ['alpha', 'beta'],
        'sentiment_llm': [0, 1],
        'extra': [9, 8],
    }
)

with patch.object(pipeline, 'clean_text_series', lambda series: series.str.upper()):
    transformed = pipeline.preprocess_dataframe(
        frame,
        text_column='content',
        output_column='clean_content',
        keep_raw=True,
        min_chars=1,
        drop_duplicates=False,
        keep_columns=['review_id', 'content_raw', 'clean_content'],
    )

display(transformed)

assert transformed.columns.tolist() == ['review_id', 'content_raw', 'clean_content']
assert transformed['review_id'].tolist() == [1, 2]
assert transformed['content_raw'].tolist() == ['alpha', 'beta']
assert transformed['clean_content'].tolist() == ['ALPHA', 'BETA']


,review_id,content_raw,clean_content
0,1,alpha,ALPHA
1,2,beta,BETA


In [43]:
with TemporaryDirectory() as tmpdir:
    tmpdir = Path(tmpdir)
    input_path = tmpdir / 'input.csv'
    output_path = tmpdir / 'output.csv'

    file_frame = pd.DataFrame({'review_id': [1], 'content': ['hello world'], 'sentiment_llm': [1]})
    file_frame.to_csv(input_path, index=False, encoding='utf-8-sig')

    cleaned = pipeline.preprocess_file(
        input_path,
        output_path,
        text_column='content',
        keep_raw=True,
        min_chars=5,
        drop_duplicates=True,
        keep_columns=['review_id', 'content_raw', 'content'],
    )

    saved = pd.read_csv(output_path)
    display(cleaned)
    display(saved)

    assert output_path.exists()
    assert cleaned.columns.tolist() == ['review_id', 'content_raw', 'content']
    assert saved.columns.tolist() == ['review_id', 'content_raw', 'content']
    assert cleaned['content_raw'].tolist() == ['hello world']
    assert saved['content_raw'].tolist() == ['hello world']
    assert cleaned['content'].tolist() == ['hello world']
    assert saved['content'].tolist() == ['hello world']


,review_id,content_raw,content
0,1,hello world,hello world


,review_id,content_raw,content
0,1,hello world,hello world


## 6. Real data sanity checks

Load the raw dataset, inspect the shape, and preview a few rows before and after cleaning.


In [44]:
raw_candidates = [
    PROJECT_ROOT / 'data' / 'raw' / 'tiki-book-review.json',
    PROJECT_ROOT / 'data' / 'interim' / 'raw_train' / 'train.json',
    PROJECT_ROOT / 'data' / 'interim' / 'raw_val' / 'val.json',
    PROJECT_ROOT / 'data' / 'interim' / 'raw_test' / 'test.json',
]
raw_path = next((path for path in raw_candidates if path.exists()), None)
if raw_path is None:
    raise FileNotFoundError('No raw dataset found in data/raw or data/interim')

raw_df = pd.read_json(raw_path)
display(summarize_frame(raw_df, 'content'))
display(raw_df.head(3))

if 'sentiment_llm' in raw_df.columns:
    display(raw_df['sentiment_llm'].value_counts(dropna=False).sort_index())

content_rows = raw_df.loc[raw_df['content'].notna(), ['review_id', 'content', 'sentiment_llm']].copy()
sample = content_rows.sample(n=min(12, len(content_rows)), random_state=42).reset_index(drop=True)
sample_preview = stepwise_preview(sample['content'], min_chars=quality_filter.SHORT_TEXT_MIN_CHARS)
sample_preview.insert(0, 'review_id', sample['review_id'])
sample_preview.insert(1, 'sentiment_llm', sample['sentiment_llm'])
display(sample_preview[['review_id', 'sentiment_llm', 'raw', 'unicode', 'noise', 'emoji', 'vocab', 'format', 'clean', 'meaningful']])

changed = sample_preview[sample_preview['raw'].astype('string') != sample_preview['clean'].astype('string')]
if not changed.empty:
    display(changed[['review_id', 'raw', 'format', 'clean', 'meaningful']])


,rows,columns,missing_text,duplicate_normalized_texts
0,13412,15,6,18


,review_id,rating,review_title,content,product_id,product_name,category,created_at,sentiment_llm,as_content,as_physical,as_price,as_packaging,as_delivery,as_service
0,20146420,5,Cực kì hài lòng,"Sách về tay đẹp, không tì vết.",190705743,Shadow and Bone: Shadow and Bone : Book 1,Children's Books,1752424584,2.0,NaN,2.0,NaN,NaN,NaN,NaN
1,20133985,5,Cực kì hài lòng,Mua cỡ 200 hoi :v chất lượng bìa okey mà spine UK hơi cứng.,190705743,Shadow and Bone: Shadow and Bone : Book 1,Children's Books,1749535073,1.0,NaN,1.0,NaN,NaN,NaN,NaN
2,18374869,5,Cực kì hài lòng,Cực kì hài lòng,190705743,Shadow and Bone: Shadow and Bone : Book 1,Children's Books,1670512641,2.0,NaN,NaN,NaN,NaN,NaN,NaN


sentiment_llm
0.0    7028
1.0    2187
2.0    4196
NaN       1
Name: count, dtype: int64

,review_id,sentiment_llm,raw,unicode,noise,emoji,vocab,format,clean,meaningful
0,20105212,0.0,Tôi đặt bookcare mà lại giao hàng ko có,Tôi đặt bookcare mà lại giao hàng ko có,Tôi đặt bookcare mà lại giao hàng ko có,Tôi đặt bookcare mà lại giao hàng ko có,Tôi đặt book_care mà lại giao hàng không có,Tôi đặt book_care mà lại giao hàng không có,tôi đặt book_care mà lại giao hàng không có,True
1,14035011,2.0,"sách đẹp, ưng ý","sách đẹp, ưng ý","sách đẹp, ưng ý","sách đẹp, ưng ý","sách đẹp, ưng ý","sách đẹp, ưng ý","sách đẹp, ưng ý",True
2,17236371,2.0,"Sách viết dễ hiểu, có tính ứng dụng cao. Nên đọc để có thể giao tiếp với mọi người tốt hơn và đạt được những điều mì...","Sách viết dễ hiểu, có tính ứng dụng cao. Nên đọc để có thể giao tiếp với mọi người tốt hơn và đạt được những điều mì...","Sách viết dễ hiểu, có tính ứng dụng cao. Nên đọc để có thể giao tiếp với mọi người tốt hơn và đạt được những điều mì...","Sách viết dễ hiểu, có tính ứng dụng cao. Nên đọc để có thể giao tiếp với mọi người tốt hơn và đạt được những điều mì...","Sách viết dễ hiểu, có tính ứng dụng cao. Nên đọc để có thể giao tiếp với mọi người tốt hơn và đạt được những điều mì...","Sách viết dễ hiểu, có tính ứng dụng cao. Nên đọc để có thể giao tiếp với mọi người tốt hơn và đạt được những điều mì...","sách viết dễ hiểu, có tính ứng dụng cao. nên đọc để có thể giao tiếp với mọi người tốt hơn và đạt được những điều mì...",True
3,19684319,2.0,"Đọc xong mới đánh giá, truyện rất hay nhé, đọc có đoạn mình rưng rưng nước mắt.","Đọc xong mới đánh giá, truyện rất hay nhé, đọc có đoạn mình rưng rưng nước mắt.","Đọc xong mới đánh giá, truyện rất hay nhé, đọc có đoạn mình rưng rưng nước mắt.","Đọc xong mới đánh giá, truyện rất hay nhé, đọc có đoạn mình rưng rưng nước mắt.","Đọc xong mới đánh giá, truyện rất hay nhé, đọc có đoạn mình rưng rưng nước mắt.","Đọc xong mới đánh giá, truyện rất hay nhé, đọc có đoạn mình rưng rưng nước mắt.","đọc xong mới đánh giá, truyện rất hay nhé, đọc có đoạn mình rưng rưng nước mắt.",True
4,9444017,2.0,"Sách mới, nội dung hay.","Sách mới, nội dung hay.","Sách mới, nội dung hay.","Sách mới, nội dung hay.","Sách mới, nội dung hay.","Sách mới, nội dung hay.","sách mới, nội dung hay.",True
5,17926066,0.0,Shop đóng gói chán thật sự 😃,Shop đóng gói chán thật sự 😃,Shop đóng gói chán thật sự 😃,Shop đóng gói chán thật sự emoji_grinning_face_with_big_eyes,Shop đóng gói chán thật sự emoji_grinning_face_with_big_eyes,Shop đóng gói chán thật sự emoji_grinning_face_with_big_eyes,shop đóng gói chán thật sự emoji_grinning_face_with_big_eyes,True
6,3433088,2.0,"Sách in đẹp, dễ hiểu, phù hợp bé 5-6 tuổi trở lên","Sách in đẹp, dễ hiểu, phù hợp bé 5-6 tuổi trở lên","Sách in đẹp, dễ hiểu, phù hợp bé 5-6 tuổi trở lên","Sách in đẹp, dễ hiểu, phù hợp bé 5-6 tuổi trở lên","Sách in đẹp, dễ hiểu, phù hợp bé 5-6 tuổi trở lên","Sách in đẹp, dễ hiểu, phù hợp bé 5-6 tuổi trở lên","sách in đẹp, dễ hiểu, phù hợp bé 5-6 tuổi trở lên",True
7,8005749,2.0,Sách rất hay và đưa ra rất nhiều điểm quan trọng của dinh dưỡng đối với vận động viên thi đấu sức bền,Sách rất hay và đưa ra rất nhiều điểm quan trọng của dinh dưỡng đối với vận động viên thi đấu sức bền,Sách rất hay và đưa ra rất nhiều điểm quan trọng của dinh dưỡng đối với vận động viên thi đấu sức bền,Sách rất hay và đưa ra rất nhiều điểm quan trọng của dinh dưỡng đối với vận động viên thi đấu sức bền,Sách rất hay và đưa ra rất nhiều điểm quan trọng của dinh dưỡng đối với vận động viên thi đấu sức bền,Sách rất hay và đưa ra rất nhiều điểm quan trọng của dinh dưỡng đối với vận động viên thi đấu sức bền,sách rất hay và đưa ra rất nhiều điểm quan trọng của dinh dưỡng đối với vận động viên thi đấu sức bền,True
8,17993171,2.0,Nội dung hữu ích và giá trị đáng mua,Nội dung hữu ích và giá trị đáng mua,Nội dung hữu ích và giá trị đáng mua,Nội dung hữu ích và giá trị đáng mua,Nội dung hữu ích và giá trị đáng mua,Nội dung hữu ích và giá trị đáng mua,nội dung hữu ích và giá trị đáng mua,True
9,18604824,2.0,"giao hàng siêu tốc luôn, tiki gi

,review_id,raw,format,clean,meaningful
0,20105212,Tôi đặt bookcare mà lại giao hàng ko có,Tôi đặt book_care mà lại giao hàng không có,tôi đặt book_care mà lại giao hàng không có,True
2,17236371,"Sách viết dễ hiểu, có tính ứng dụng cao. Nên đọc để có thể giao tiếp với mọi người tốt hơn và đạt được những điều mì...","Sách viết dễ hiểu, có tính ứng dụng cao. Nên đọc để có thể giao tiếp với mọi người tốt hơn và đạt được những điều mì...","sách viết dễ hiểu, có tính ứng dụng cao. nên đọc để có thể giao tiếp với mọi người tốt hơn và đạt được những điều mì...",True
3,19684319,"Đọc xong mới đánh giá, truyện rất hay nhé, đọc có đoạn mình rưng rưng nước mắt.","Đọc xong mới đánh giá, truyện rất hay nhé, đọc có đoạn mình rưng rưng nước mắt.","đọc xong mới đánh giá, truyện rất hay nhé, đọc có đoạn mình rưng rưng nước mắt.",True
4,9444017,"Sách mới, nội dung hay.","Sách mới, nội dung hay.","sách mới, nội dung hay.",True
5,17926066,Shop đóng gói chán thật sự 😃,Shop đóng gói chán thật sự emoji_grinning_face_with_big_eyes,shop đóng gói chán thật sự emoji_grinning_face_with_big_eyes,True
6,3433088,"Sách in đẹp, dễ hiểu, phù hợp bé 5-6 tuổi trở lên","Sách in đẹp, dễ hiểu, phù hợp bé 5-6 tuổi trở lên","sách in đẹp, dễ hiểu, phù hợp bé 5-6 tuổi trở lên",True
7,8005749,Sách rất hay và đưa ra rất nhiều điểm quan trọng của dinh dưỡng đối với vận động viên thi đấu sức bền,Sách rất hay và đưa ra rất nhiều điểm quan trọng của dinh dưỡng đối với vận động viên thi đấu sức bền,sách rất hay và đưa ra rất nhiều điểm quan trọng của dinh dưỡng đối với vận động viên thi đấu sức bền,True
8,17993171,Nội dung hữu ích và giá trị đáng mua,Nội dung hữu ích và giá trị đáng mua,nội dung hữu ích và giá trị đáng mua,True
9,18604824,"giao hàng siêu tốc luôn, tiki giao hàng uy tín. đủ hàng và đẹp lắm nha. nên mua nha mn","giao hàng siêu tốc luôn, tiki giao hàng uy tín. đủ hàng và đẹp lắm nha. nên mua nha mọi_người","giao hàng siêu tốc luôn, tiki giao hàng uy tín. đủ hàng và đẹp lắm nha. nên mua nha mọi_người",True
10,9787706,"Êu ơi, đặt hàng tiki bao lâu nay mà đợt này tiki làm mình thất vọng quá, tiki đóng hộp quá là lỏng lẻo luôn ấy, đến ...","Êu ơi, đặt hàng tiki bao lâu nay mà đợt này tiki làm mình thất vọng quá, tiki đóng hộp quá là lỏng lẻo luôn ấy, đến ...","êu ơi, đặt hàng tiki bao lâu nay mà đợt này tiki làm mình thất vọng quá, tiki đóng hộp quá là lỏng lẻo luôn ấy, đến ...",True


## 7. Split leakage check

This reproduces `test_split_dataset.py` and confirms duplicate content stays in one split.


In [45]:
split_frame = pd.DataFrame(
    {
        'review_id': list(range(1, 15)),
        'content': [
            'Alpha product',
            'Alpha   product ',
            'Bravo product',
            'bravo product',
            'Charlie product',
            'Delta product',
            'Echo product',
            'Foxtrot product',
            'Golf product',
            'Hotel product',
            'India product',
            'Juliet product',
            'Kilo product',
            'Lima product',
        ],
        'sentiment_llm': [0, 0, 1, 1, 2, 0, 1, 2, 0, 1, 2, 0, 1, 2],
    }
)

with patch.object(split_dataset, 'clean_text_series', lambda series: series):
    raw_train, raw_val, raw_test = split_dataset._split_raw_rows(split_frame)

train_keys = {quality_filter.normalize_for_duplicate(value) for value in raw_train['content']}
val_keys = {quality_filter.normalize_for_duplicate(value) for value in raw_val['content']}
test_keys = {quality_filter.normalize_for_duplicate(value) for value in raw_test['content']}

split_overview = pd.DataFrame(
    {
        'split': ['train', 'val', 'test'],
        'rows': [len(raw_train), len(raw_val), len(raw_test)],
        'unique_keys': [len(train_keys), len(val_keys), len(test_keys)],
    }
)
display(split_overview)
display(pd.DataFrame([
    {
        'train_val_overlap': len(train_keys & val_keys),
        'train_test_overlap': len(train_keys & test_keys),
        'val_test_overlap': len(val_keys & test_keys),
    }
]))

assert train_keys.isdisjoint(val_keys)
assert train_keys.isdisjoint(test_keys)
assert val_keys.isdisjoint(test_keys)

duplicate_key = quality_filter.normalize_for_duplicate('Alpha product')
membership = sum(duplicate_key in split_keys for split_keys in [train_keys, val_keys, test_keys])
assert membership == 1


,split,rows,unique_keys
0,train,9,8
1,val,3,2
2,test,2,2


,train_val_overlap,train_test_overlap,val_test_overlap
0,0,0,0
